# Task 3 - Kafka topic and event design

Nodes, edges, metadata, and parser failures have separate topics. Every record carries a schema version, event time, stable key, content hash, run ID, event ID, and operation.

In [1]:
import subprocess
from pathlib import Path

root = Path('..').resolve()
topics = [
    'cpg.nodes.v1', 'cpg.edges.v1', 'cpg.source-metadata.v1',
    'cpg.parser-errors.v1', 'cpg.neo4j-dlq.v1',
]
for topic in topics:
    result = subprocess.run(
        ['docker', 'compose', 'exec', '-T', 'broker', 'kafka-topics',
         '--bootstrap-server', 'broker:29092', '--describe', '--topic', topic],
        cwd=root, capture_output=True, text=True, check=True,
    )
    print(result.stdout.rstrip())
print('PASS: four required topics and the connector DLQ are configured')

Topic: cpg.nodes.v1	TopicId: YyY-414WRLSQ9NjuaZCY1A	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.nodes.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.edges.v1	TopicId: tQC-F1ksTbm5HvcTgilxjQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.edges.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.source-metadata.v1	TopicId: 1YXULaXYTue71lwhS8p7og	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=compact
	Topic: cpg.source-metadata.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.parser-errors.v1	TopicId: 0MJGt2teQquWtGdKsh0UUQ	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=delete,retention.ms=604800000
	Topic: cpg.parser-errors.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:


Topic: cpg.neo4j-dlq.v1	TopicId: 8DQjWsITTjuKPshAYrF4BA	PartitionCount: 1	ReplicationFactor: 1	Configs: cleanup.policy=delete,retention.ms=604800000
	Topic: cpg.neo4j-dlq.v1	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr:
PASS: four required topics and the connector DLQ are configured


In [2]:
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

root = Path('..').resolve()
sys.path.insert(0, str(root))
from cpg_parser.manifest import Manifest
from cpg_parser.publisher import MemoryPublisher
from cpg_parser.service import ParserService

samples = {}
with TemporaryDirectory() as directory:
    valid_publisher = MemoryPublisher()
    with Manifest(Path(directory) / 'valid.sqlite') as manifest:
        ParserService(root / 'source-repo', 'huggingface/optimum', valid_publisher, manifest).process(
            'optimum/version.py', force=True
        )
    for topic, key, value in valid_publisher.records:
        samples.setdefault(topic, {'key': key, 'value': value})

    error_publisher = MemoryPublisher()
    with Manifest(Path(directory) / 'error.sqlite') as manifest:
        ParserService(
            root / 'tests/fixtures/invalid_repo', 'fixture/invalid', error_publisher, manifest
        ).process('broken.py', force=True)
    for topic, key, value in error_publisher.records:
        if topic == 'cpg.parser-errors.v1':
            samples[topic] = {'key': key, 'value': value}

required = {'cpg.nodes.v1', 'cpg.edges.v1', 'cpg.source-metadata.v1', 'cpg.parser-errors.v1'}
assert required <= set(samples)
for record in samples.values():
    assert record['key']
    assert record['value']['schema_version'] == '1.0'
    assert record['value']['event_time'].endswith('Z')
print(json.dumps(samples, indent=2, ensure_ascii=False))
print('PASS: node, edge, metadata, and parser-error event samples validated')

{
  "cpg.nodes.v1": {
    "key": "b48918aa8c75bba8b82275c2b6b971a89a85434bd0cdcd8d02fc9cd0b59302e8",
    "value": {
      "schema_version": "1.0",
      "event_time": "2026-07-22T05:22:45.398796Z",
      "repo_id": "huggingface/optimum",
      "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
      "run_id": "22a94f524f964410abe1d62f372e54d8",
      "content_hash": "079eca803ea5bfe068bc805997b013cc14c9ddc8629a2ec634d12bcebb6720ce",
      "op": "upsert",
      "node": {
        "id": "b48918aa8c75bba8b82275c2b6b971a89a85434bd0cdcd8d02fc9cd0b59302e8",
        "kind": "AST",
        "ast_type": "Module",
        "file_id": "867c06c1f6e594bef9a16137312503a0870399626a16dded522ce7a24143b1a7",
        "structural_path": "$",
        "line": 0,
        "column": 0,
        "end_line": 0,
        "end_column": 0,
        "name": "",
        "value": ""
      },
      "event_id": "63b8fd885ad93687b36df566474246bd19b54e511ca2fa80404964934a87d076"
    }
  },
  "cpg.edg

## Reflection

Compaction retains current keyed state but does not alone guarantee database idempotency. Stable keys, Neo4j `MERGE`, MongoDB replacement upserts, and the parser manifest provide the remaining guarantees.